# M00 · Course Setup, Computational Environment & Orientation
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M00_Setup/M00_Setup_and_Orientation.ipynb)

⏱ ~60 min · Difficulty: ★☆☆ · Prerequisites: None


## 1. Learning Outcomes

By the end of this orientation notebook, you will be able to:
1. **Explain** why programming, data structures, and automation are indispensable for modern high-throughput computational biology and genomics.
2. **Configure and verify** a reproducible Python 3 scientific computing environment locally (using Conda or virtual environments) and in Google Colab.
3. **Navigate** the Jupyter Notebook ecosystem effectively, utilizing modal keyboard shortcuts, kernel lifecycle controls, inline documentation (`?` and `help()`), and IPython magic commands (`%timeit`, `%%writefile`).
4. **Inspect and traverse** the course repository directory structure (`data/`, `modules/`, `outputs/`, `assessments/`, `cheatsheets/`) while adhering to strict input/output separation principles.
5. **Execute** an end-to-end environment diagnostic check verifying core Python packages and benchmarking basic sequence operations.
6. **Decode** high-throughput sequencing quality representations (Phred scores) and safely parse file system paths.


## 2. Prerequisites

- Basic computer literacy: navigating folders, opening a terminal/command prompt, and using a modern web browser.
- **No prior programming experience in Python is required.** This course builds from computational zero to advanced bioinformatics applications.


## 3. Setup

The following cell detects your execution environment (Google Colab vs. local machine), configures relative paths to the course repository, and ensures required output folders exist. Run it first!


In [ ]:
# --- Setup & Environment Verification ----------------------------------
import sys
import os
import platform
from pathlib import Path

# Detect whether we are executing inside Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Setting up repository clone...")
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root: search upwards until 'data' folder is found
    p = Path.cwd()
    found_root = None
    for candidate in [p] + list(p.parents):
        if (candidate / "data" / "sequences").exists() or (candidate / "COURSE_PLAN.md").exists():
            found_root = candidate
            break
    if found_root:
        ROOT = found_root
    else:
        repo_candidate = Path("/Users/prateekpaul/Library/CloudStorage/GoogleDrive-prateekp@iiitd.ac.in/My Drive/IIITD/mygithub/ProgrammingCB")
        ROOT = repo_candidate if repo_candidate.exists() else p

# Define canonical path references
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / "M00"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Python version     : {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"✓ Platform           : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"✓ Repository Root    : {ROOT.resolve()}")
print(f"✓ Data Directory     : {DATA_DIR.resolve()}")
print(f"✓ Output Directory   : {OUT_DIR.resolve()}")


## 4. Concept Sections

---

### 4.1 The Biological Imperative for Computation

#### 4.1.1 Theory: The Scale of Modern Biology
Biological inquiry has undergone an irreversible phase transition: from an observational, low-throughput wet-lab discipline to an information-dense, quantitative computational science.

```
┌────────────────────────────────────────────────────────────────────────┐
│                   THE GENOMIC DATA EXPLOSION                           │
├──────────────────────────┬─────────────────────────────────────────────┤
│ Human Genome Size        │ 3.2 billion base pairs (haploid)            │
│ Illumina NovaSeq X Run   │ Up to 16–20 billion reads (~6 Terabytes)   │
│ NCBI Sequence Read Arch. │ Exceeds 35 Petabytes of raw sequencing data │
│ AlphaFold DB             │ >200 million predicted 3D protein structures│
└──────────────────────────┴─────────────────────────────────────────────┘
```

Manual analysis using spreadsheet software (such as Microsoft Excel) is not only impractical at this scale—it introduces catastrophic errors. 
> [!WARNING] The Excel Gene Symbol Scandal
> In a landmark audit published in *Genome Biology* (Ziemann et al., 2016), researchers analyzed supplementary Excel files across 3,597 published scientific genomics papers. They discovered that **19.6% of papers contained severe gene name corruptions**. Excel had automatically converted gene symbols into calendar dates (e.g., `SEPT2` [Septin 2] converted into `2-Sep`, and `MARCH1` [Membrane Associated Ring-CH-Type Finger 1] converted into `1-Mar`).
> 
> Writing deterministic, programmatic scripts in Python ensures **100% reproducibility, auditability, and immutability** of your raw experimental data.


In [ ]:
# Biological Motivation: Demonstrating data scale in memory
human_genome_bp = 3_200_000_000

# If stored as uncompressed ASCII characters (1 byte per nucleotide):
size_in_bytes = human_genome_bp * 1
size_in_mb = size_in_bytes / (1024 ** 2)
size_in_gb = size_in_bytes / (1024 ** 3)

print(f"Human genome (raw characters):")
print(f"  {human_genome_bp:,} base pairs")
print(f"  = {size_in_mb:,.1f} Megabytes")
print(f"  = {size_in_gb:.2f} Gigabytes (uncompressed per single genome)")

# At 30x clinical sequencing depth, raw FASTQ reads require ~90 GB per patient!
coverage = 30
print(f"  At {coverage}x depth: ~{size_in_gb * coverage:.1f} GB of raw sequencing reads.")


#### 4.1.2 ✏️ Try it: Calculate Sequencing Data Scale
Calculate the storage in Gigabytes required to store the sequencing reads for a cohort of 50 cancer patients sequenced at 50x coverage (assume 1 byte per base pair on a 3.2 Gb genome).


In [ ]:
# Calculate required storage in Gigabytes
patients = 50
depth = 50
genome_size = 3.2e9

# Total bases sequenced
total_bases = patients * depth * genome_size
total_gb = total_bases / (1024 ** 3)

print(f"Total sequencing volume for {patients} patients at {depth}x: {total_gb:,.2f} GB ({total_gb/1024:.2f} TB)")
assert 7000 < total_gb < 8000, "Calculation mismatch!"


#### 4.1.3 ⚠️ Common Mistakes in Data Scale
- **Confusing bits and bytes**: In genomics, base pairs are often stored as 1 byte (8 bits) in plain text, but can be packed into 2 bits per base ($A=00, C=01, G=10, T=11$) in binary BAM/CRAM formats.
- **Underestimating RAM requirements**: Loading a 10 GB file directly into memory using a naive `file.read()` will crash your machine if available RAM is insufficient. Later modules will teach streaming and line-by-line generators.


---

### 4.2 Python Environments: Conda, venv, and Google Colab

#### 4.2.1 Theory: Why Virtual Environments?
In computational biology, different tools and publications often rely on conflicting library versions. For example, a legacy phylogenetic package may require Python 3.8 and NumPy 1.20, whereas an AlphaFold or PyTorch pipeline requires Python 3.11 and NumPy 2.x.

A **virtual environment** is an isolated directory on your machine containing a dedicated Python interpreter and an independent set of packages.

| Environment Manager | Best Used For | Notes |
|---|---|---|
| **Miniconda / Conda** | Bioinformatics, cross-language tools (C/C++ binaries, BLAST, SAMtools) | Recommended for local scientific work via Bioconda. |
| **Python venv** | Pure Python projects, web servers | Lightweight, built into Python standard library. |
| **Google Colab** | Zero-install interactive exploration, free GPUs/TPUs | Cloud-hosted; session resets after idle timeout. |


In [ ]:
# Inspecting Python environment details
import sys
import platform

print("Active Python Executable:", sys.executable)
print("Python Version Tuple    :", sys.version_info[:3])
print("Platform Architecture   :", platform.architecture()[0])

assert sys.version_info.major == 3, "Python 3 is strictly required!"
print("✓ Environment sanity check passed: Running Python 3.")


#### 4.2.2 ⚠️ Common Environment Mistakes
- **Installing packages into the 'base' conda environment**: Always create a dedicated environment per project (`conda create -n biocomp python=3.11`).
- **Mixing pip and conda blindly**: When using Conda, install available packages using `conda install` first. Only use `pip` for packages unavailable on Bioconda/conda-forge.
- **Forgetting to activate your environment**: Running `python script.py` in a new terminal without `conda activate biocomp` will execute your system Python, causing missing package errors.


---

### 4.3 Jupyter Notebook Essentials for Biologists

#### 4.3.1 Theory: The Interactive Computing Paradigm
Jupyter Notebooks combine narrative Markdown, executable code, and visual graphics.

```
┌────────────────────────────────────────────────────────┐
│             JUPYTER MODAL EDITING SYSTEM               │
├──────────────────────────┬─────────────────────────────┤
│ Command Mode (Blue line) │ Press [Esc] to enter        │
│                          │ Controls notebook structure │
│                          │ Keys: A (above), B (below), │
│                          │ D,D (delete), M (markdown), │
│                          │ Y (code), Enter (edit mode) │
├──────────────────────────┼─────────────────────────────┤
│ Edit Mode (Green line)   │ Press [Enter] on a cell     │
│                          │ Typing code or markdown text│
├──────────────────────────┼─────────────────────────────┤
│ Execution Shortcuts      │ [Shift] + [Enter] : Run & go│
│                          │ [Ctrl]  + [Enter] : Run stay│
└──────────────────────────┴─────────────────────────────┘
```

#### Magic Commands
IPython provides specialized commands prefixed by `%` (line magics) or `%%` (cell magics):
- `%timeit`: Micro-benchmarks code execution over multiple loops.
- `%%writefile <path>`: Writes the entire contents of a cell directly to a file on disk.


In [ ]:
# Demonstrating %timeit for micro-benchmarking
codons = ["ATG", "GCG", "TAC", "TTC", "GAA"] * 100

def concat_loop(items):
    res = ""
    for item in items:
        res += item
    return res

def concat_join(items):
    return "".join(items)

assert concat_loop(codons) == concat_join(codons)

print("Timing loop concatenation:")
%timeit concat_loop(codons)

print("Timing str.join concatenation (idiomatic Python):")
%timeit concat_join(codons)


In [ ]:
# Writing a test FASTA file to the output folder using Python file I/O
test_fasta_path = OUT_DIR / "sample_primer.fasta"

fasta_content = """>primer_fwd_tp53
ATGGAGGAGCCGCAGTCAGAT
>primer_rev_tp53
GTCTGAGTCCTGCCCTTCTGT
"""

with open(test_fasta_path, "w") as f:
    f.write(fasta_content)

print(f"Successfully generated test file at: {test_fasta_path}")
assert test_fasta_path.exists()
assert test_fasta_path.stat().st_size > 0


---

### 4.4 Course Repository Structure & Data Flow

#### 4.4.1 Theory: Clean Project Architecture
In data-intensive biology, reproducibility requires separating **immutable input data** from **transient output results** and **modular code**.

```
ProgrammingCB/
├── data/                    # IMMUTABLE RAW DATA (never write here!)
│   ├── sequences/           # FASTA, FASTQ, raw nucleotide/protein files
│   ├── tables/              # TSV/CSV metadata, coordinates, counts
│   ├── structures/          # PDB, mmCIF macromolecular files
│   ├── genbank/             # Rich GenBank records
│   └── html/                # Sample biological HTML pages
├── modules/                 # COURSE NOTEBOOKS (M00 to M06)
├── outputs/                 # GENERATED ARTIFACTS (plots, tables, fasta)
│   └── MXX/                 # One sub-directory per module
├── assessments/             # Quizzes and practice tests
├── cheatsheets/             # Concise reference sheets
├── requirements.txt         # Pip dependency manifest
└── environment.yml          # Conda environment definition
```

> [!IMPORTANT] The Golden Rule of Course Data
> Never write, modify, or delete files inside the `data/` directory. All programmatically generated files must be written to `outputs/<Module_Name>/`.


In [ ]:
# Repository Directory Verification
print("Repository Structure Audit:")
for subdir in ["sequences", "tables", "structures", "genbank"]:
    target = DATA_DIR / subdir
    if target.exists():
        files = list(target.glob("*"))
        print(f"  📁 data/{subdir:<12} : {len(files)} files present (e.g. {[f.name for f in files[:2]]})")
    else:
        print(f"  ⚠️ data/{subdir:<12} : Directory not yet populated")


---

### 4.5 Git in 10 Minutes: Practical Version Control for Biologists

#### 4.5.1 Theory: Why Version Control?
Modern scientific code must be verifiable, auditable, and shareable. Git tracks incremental snapshots ("commits") of your codebase.

```
Working Directory   ──( git add )──>   Staging Area   ──( git commit )──>   Local Repository   ──( git push )──>   GitHub Remote
```

#### Essential Commands:
1. `git clone <url>`: Download a complete copy of the remote repository and commit history.
2. `git status`: Show modified, staged, and untracked files.
3. `git add <file>`: Stage changes for inclusion in the next commit snapshot.
4. `git commit -m "Descriptive message"`: Permanently record staged snapshot to history.
5. `git pull`: Fetch and integrate changes from GitHub.
6. `git push`: Upload local commits to your GitHub fork or repository.


In [ ]:
# Checking Git availability from Python
import subprocess

try:
    git_version = subprocess.check_output(["git", "--version"], text=True).strip()
    print("Git detected:", git_version)
except Exception as e:
    print("Git command not found or not in PATH:", e)


## 5. Practice Exercises

Complete each exercise below. Exercise cells contain assertions that automatically validate your implementation.


### Exercise 1 (Level 1 ★): Diagnostic Environment & Scientific Stack Verification

**Problem**: Verify that all core Python standard library modules required for sequence analysis (`math`, `re`, `collections`, `itertools`, `pathlib`, `json`, `platform`) can be imported without error. Confirm that the current Python interpreter version is >= 3.8.

- **Input**: Current runtime environment.
- **Expected Output**: A dictionary reporting status `"OK"` for each module and boolean `True` for version compatibility.


In [ ]:
def verify_environment() -> dict:
    """Checks core standard library modules and verifies Python version >= 3.8."""
    modules_to_test = ['math', 're', 'collections', 'itertools', 'pathlib', 'json', 'platform']
    results = {}
    for mod in modules_to_test:
        try:
            __import__(mod)
            results[mod] = "OK"
        except ImportError:
            results[mod] = "FAIL"
            
    is_supported_py = (sys.version_info.major == 3) and (sys.version_info.minor >= 8)
    results["python_version_supported"] = is_supported_py
    results["python_version_string"] = platform.python_version()
    return results

env_report = verify_environment()
for k, v in env_report.items():
    print(f"  {k:<26}: {v}")

assert env_report["python_version_supported"] is True, "Python version must be >= 3.8!"
for mod in ['math', 're', 'collections', 'itertools', 'pathlib', 'json', 'platform']:
    assert env_report[mod] == "OK", f"Module {mod} failed to import!"
print("✓ Exercise 1 passed successfully!")


### Exercise 2 (Level 1 ★): Reading Sequence Data and Writing Output Logs

**Problem**: 
1. Read the synthetic gene sequence located at `data/sequences/toy_gene_with_introns.txt`.
2. Strip trailing newlines or whitespace.
3. Compute total sequence length in nucleotides.
4. Verify that the sequence starts with canonical start codon `ATG`.
5. Write an audit log file to `outputs/M00/gene_audit.txt` in format:
   `LENGTH=<length>|STARTS_WITH_ATG=<True/False>`

- **Input**: `toy_gene_with_introns.txt`
- **Expected Output**: Written log file in `outputs/M00/` and verified assertions.


In [ ]:
gene_path = DATA_DIR / "sequences" / "toy_gene_with_introns.txt"
log_path = OUT_DIR / "gene_audit.txt"

with open(gene_path, "r") as f:
    seq_data = f.read().strip()

seq_len = len(seq_data)
starts_atg = seq_data.startswith("ATG")

with open(log_path, "w") as f:
    f.write(f"LENGTH={seq_len}|STARTS_WITH_ATG={starts_atg}\n")

print(f"Gene length: {seq_len} bp | Starts with ATG: {starts_atg}")
print(f"Audit log created at: {log_path}")

assert log_path.exists(), "Output log file was not created!"
with open(log_path, "r") as f:
    log_content = f.read().strip()

assert f"LENGTH={seq_len}" in log_content
assert "STARTS_WITH_ATG=True" in log_content
assert seq_len > 200, f"Expected gene length > 200 bp, got {seq_len}"
print("✓ Exercise 2 passed successfully!")


### Exercise 3 (Level 2 ★★): Micro-benchmarking Sequence Algorithms

**Problem**: 
Implement two distinct Python functions to compute total Guanine (`G`) and Cytosine (`C`) count of a nucleotide string:
1. `count_gc_loop(seq)`: Iterates character-by-character using an explicit `for` loop.
2. `count_gc_builtin(seq)`: Uses Python's built-in string methods (`seq.count('G') + seq.count('C')`).

Generate a synthetic sequence of length 100,000 nucleotides. Measure execution time using `time.perf_counter()`. Verify both functions return the exact same count, and assert that the built-in C-optimized method is at least $1.5\times$ faster.


In [ ]:
import time
import random

random.seed(42)
test_dna = "".join(random.choices("ACGT", k=100_000))

def count_gc_loop(seq: str) -> int:
    count = 0
    for char in seq:
        if char == "G" or char == "C":
            count += 1
    return count

def count_gc_builtin(seq: str) -> int:
    return seq.count("G") + seq.count("C")

t0 = time.perf_counter()
res_loop = count_gc_loop(test_dna)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
res_builtin = count_gc_builtin(test_dna)
t_builtin = time.perf_counter() - t0

speedup = t_loop / t_builtin if t_builtin > 0 else 1.0

print(f"Loop implementation    : {res_loop:,} bases in {t_loop*1000:.3f} ms")
print(f"Built-in implementation: {res_builtin:,} bases in {t_builtin*1000:.3f} ms")
print(f"Speedup factor         : {speedup:.1f}x faster using built-in methods")

assert res_loop == res_builtin, "Count results do not match!"
assert res_loop > 0, "Count should be positive!"
assert speedup >= 1.5, f"Expected built-in method to be faster (got {speedup:.1f}x)"
print("✓ Exercise 3 passed successfully!")


### Exercise 4 (Level 2 ★★): Safe File System Path Inspection and Format Classification

**Biological Background**: In automated genomic workflows, scripts must inspect input paths, verify file existence, and categorize file formats (e.g. FASTA, FASTQ, TSV, GenBank) based on file extensions before dispatching to specialized parsers.

**Task**: Write a function `classify_biological_file(path_obj: Path) -> dict` that:
1. Checks if `path_obj` exists.
2. If it does not exist, returns `{"exists": False, "format": "Missing", "size_bytes": 0}`.
3. If it exists, determines the biological format based on its file extension:
   - `'.fasta'` or `'.fa'` or `'.fna'`: `'FASTA'`
   - `'.fastq'` or `'.fq'`: `'FASTQ'`
   - `'.tsv'` or `'.csv'`: `'Table'`
   - `'.gb'` or `'.gbk'` or `'.gp'`: `'GenBank'`
   - `'.pdb'` or `'.cif'`: `'Macromolecular Structure'`
   - Any other suffix: `'Generic Text'`
4. Returns a dictionary containing `'exists'`, `'format'`, and `'size_bytes'` (`path_obj.stat().st_size`).


In [ ]:
def classify_biological_file(path_obj: Path) -> dict:
    """Inspects a file path and determines its biological data format."""
    if not path_obj.exists():
        return {"exists": False, "format": "Missing", "size_bytes": 0}
        
    suffix = path_obj.suffix.lower()
    format_map = {
        '.fasta': 'FASTA', '.fa': 'FASTA', '.fna': 'FASTA',
        '.fastq': 'FASTQ', '.fq': 'FASTQ',
        '.tsv': 'Table', '.csv': 'Table',
        '.gb': 'GenBank', '.gbk': 'GenBank', '.gp': 'GenBank',
        '.pdb': 'Macromolecular Structure', '.cif': 'Macromolecular Structure'
    }
    file_format = format_map.get(suffix, 'Generic Text')
    size = path_obj.stat().st_size
    
    return {"exists": True, "format": file_format, "size_bytes": size}

# Test on real files from the repository
gene_path = DATA_DIR / "sequences" / "toy_gene_with_introns.txt"
tsv_path = DATA_DIR / "tables" / "toy_exon_coordinates.tsv"
missing_path = DATA_DIR / "sequences" / "non_existent_file.fasta"

res_gene = classify_biological_file(gene_path)
res_tsv = classify_biological_file(tsv_path)
res_miss = classify_biological_file(missing_path)

print("Classification results:")
print("  Gene file :", res_gene)
print("  TSV file  :", res_tsv)
print("  Missing   :", res_miss)

assert res_gene["exists"] is True
assert res_gene["format"] == "Generic Text" # .txt extension
assert res_tsv["exists"] is True
assert res_tsv["format"] == "Table"
assert res_miss["exists"] is False
assert res_miss["format"] == "Missing"
print("✓ Exercise 4 passed successfully!")


### Exercise 5 (Level 3 ★★★): Phred Quality Score Decoder

**Biological Background**: In Next-Generation Sequencing (Illumina, PacBio, Nanopore), base calls are accompanied by confidence estimates called **Phred Quality Scores ($Q$)**.
The Phred score is logarithmically related to the probability $P$ that a given base call is erroneous:
$$Q = -10 \log_{10}(P)$$

- $Q = 10 \implies P = 0.1$ (1 in 10 error rate, 90% accuracy)
- $Q = 20 \implies P = 0.01$ (1 in 100 error rate, 99% accuracy)
- $Q = 30 \implies P = 0.001$ (1 in 1,000 error rate, 99.9% accuracy — standard benchmark)
- $Q = 40 \implies P = 0.0001$ (1 in 10,000 error rate, 99.99% accuracy)

In FASTQ files, Phred scores ($0$ to $93$) are compactly encoded as single ASCII characters using the **Phred+33 (Sanger/Illumina 1.8+) offset**:
$$Q = \text{ord}(char) - 33$$
$$\text{Character} = \text{chr}(Q + 33)$$

For example, ASCII `'I'` has ordinal value 73, so its Phred score is $73 - 33 = 40$.

**Task**: Write two complementary functions:
1. `encode_phred_scores(q_scores: list[int]) -> str`: Converts a list of integer quality scores into an ASCII Phred+33 string.
2. `decode_phred_scores(phred_str: str) -> list[int]`: Converts an ASCII Phred+33 string back into a list of integer quality scores.
3. Compute the mean quality score of the read and verify assertions.


In [ ]:
def encode_phred_scores(q_scores: list[int]) -> str:
    """Encodes integer quality scores into Phred+33 ASCII string."""
    return "".join(chr(q + 33) for q in q_scores)

def decode_phred_scores(phred_str: str) -> list[int]:
    """Decodes Phred+33 ASCII string into integer quality scores."""
    return [ord(c) - 33 for c in phred_str]

def mean_phred_score(phred_str: str) -> float:
    """Calculates average quality score across a read."""
    scores = decode_phred_scores(phred_str)
    return round(sum(scores) / len(scores), 2) if scores else 0.0

# Demonstrate with standard Illumina scores:
# Q=40 ('I'), Q=30 ('?'), Q=20 ('5'), Q=10 ('+')
test_scores = [40, 40, 30, 20, 10]
encoded_str = encode_phred_scores(test_scores)
decoded_back = decode_phred_scores(encoded_str)
mean_q = mean_phred_score(encoded_str)

print(f"Original Q scores : {test_scores}")
print(f"Encoded FASTQ str : '{encoded_str}'")
print(f"Decoded Q scores  : {decoded_back}")
print(f"Mean read quality : Q{mean_q}")

assert encoded_str == "II?5+"
assert decoded_back == test_scores
assert mean_q == 28.0
print("✓ Exercise 5 passed successfully!")


## 6. Summary & Cheat Sheet Table

| Domain | Command / Syntax | Action / Description |
|---|---|---|
| **Jupyter** | `Esc` / `Enter` | Toggle Command mode (blue) vs Edit mode (green) |
| **Jupyter** | `Shift + Enter` | Execute current cell and advance to next |
| **Jupyter** | `A` / `B` | Insert new cell Above / Below (in Command mode) |
| **Jupyter** | `D, D` | Delete selected cell (in Command mode) |
| **Jupyter** | `M` / `Y` | Change cell type to Markdown / Code |
| **IPython** | `%timeit <stmt>` | Benchmark single-line statement execution time |
| **IPython** | `?object` or `help(obj)` | Inspect documentation and docstring of object |
| **Conda** | `conda create -n cb python=3.11` | Create new isolated environment |
| **Conda** | `conda activate cb` | Activate virtual environment |
| **Conda** | `conda env export > environment.yml`| Export reproducible environment specification |
| **Git** | `git status` | Inspect modified and untracked files |
| **Git** | `git pull` | Synchronize and fetch latest updates from remote repository |
| **FASTQ / Phred** | `ord(c) - 33` | Decode Phred+33 ASCII character to integer error score $Q$ |
| **Workflow** | `DATA_DIR` vs `OUT_DIR` | Read strictly from `data/`; write artifacts to `outputs/` |


## 7. Further Reading & References

1. **Ziemann, M., Eren, Y., & El-Osta, A. (2016)**. Gene name errors are still widespread in the scientific literature. *Genome Biology*, 17(1), 177. [DOI: 10.1186/s13059-016-1044-7](https://doi.org/10.1186/s13059-016-1044-7).
2. **Ewing, B., & Green, P. (1998)**. Base-calling of automated sequencer traces using phred. II. Error probabilities. *Genome Research*, 8(3), 186-194. (The original Phred score paper).
3. **Jupyter Project Documentation**: Interactive computing and notebook architecture. https://jupyter.org/documentation
4. **Conda & Bioconda**: The package manager for bioinformatics. https://bioconda.github.io/
5. **Real Python**: *Jupyter Notebook: An Introduction*. https://realpython.com/jupyter-notebook-introduction/
6. **Software Carpentry**: *The Unix Shell and Version Control with Git*. https://software-carpentry.org/lessons/
